# Synthetic Rank-One Evaluation of the SSL Encoder Shareability Metric 

In [1]:
import numpy as np
import torch
import copy

In [2]:
torch.manual_seed(42)
rng = np.random.default_rng(42)

## Shareability Metric

The SSL encoder shareability score is computed from the cross-covariance matrix

$$
M = \frac{1}{n} y^\top x.
$$

The shared encoder optimum uses only the symmetric part of M

$$
S = \frac{M + M^\top}{2}.
$$

The shared encoder optimum is 

$$
J_{shared}^* = max_i|\lambda_i(S)|.
$$

The seperate encoder optimum is 

$$
J_{sep}^* = \sigma_1(M).
$$

Giving the shareability metric

$$
p(M) = \frac{J_{shared}^*}{J_{sep}^*} = \frac{max_i|\lambda_i(S)|}{\sigma_1(M)}.
$$


In [3]:
def ssl_encoder_shareability(observation_1, observation_2):
        M = (observation_2.T @ observation_1) / observation_1.shape[0]
        S = (M + M.T) / 2
        
        S_eigenvalues = np.linalg.eigvalsh(S)
        J_shared = np.max(np.abs(S_eigenvalues))
        
        J_sep = np.linalg.svd(M, compute_uv=False)[0]
        
        P = J_shared / J_sep
        
        return P

## Shared Encoder

The linear shared encoder is given by

$$
z_x = w^\top x \qquad z_y = w^\top y.
$$

In [4]:
class SharedEncoder(torch.nn.Module):
    def __init__(self, vector_size):
        super().__init__()
        self.w = torch.nn.Linear(vector_size, 1, bias=False)
        
    def forward(self, x):
        return self.w(x)

## Seperate Encoder

The linear seperate encoder is given by 

$$
z_x = a^\top x \qquad z_y = b^\top y.
$$

In [5]:
class SeperateEncoder(torch.nn.Module):
    def __init__(self, vector_size):
        super().__init__()
        self.a = torch.nn.Linear(vector_size, 1, bias=False)
        self.b = torch.nn.Linear(vector_size, 1, bias=False)
        
    def forward(self, x, y):
        return self.a(x), self.b(y)

## Synthetic Rank-One Data Generation

The base observation, 'observation_1', consists of 5000 samples in $\mathbb{R}^2$ generated using the fixed random seed 42.

Three rank-one mappings are constructed to represent low, intermediate, and high degrees of encoder shareability.

The rank-one shareability metric is derived as 

$$
p(M) = \frac{1 + |u^\top v|}{2}.
$$

where $|u^\top v|$ measures the alignment between the unit vectors $u$ and $v$.

For the low shareability case, the left and right unit direction vectors are orthogonal such that

$$
u_{low}^\top v_{low} = 0,
$$

wwhich gives the minimum rank-one shareability 

$$
p(M_{low}) = \frac{1 + 0}{2} = 0.5.
$$

For the intermediate shareability case, the left and right unit direction vectors are partially aligned such that

$$
u_{low}^\top v_{low} = 0.5,
$$

which gives the intermediate rank-one shareability 

$$
p(M_{mid}) = \frac{1 + 0.5}{2} = 0.75.
$$

For the high shareability case, the left and right unit direction vectors are fully aligned such that

$$
u_{low}^\top v_{low} = 1,
$$

which gives the maximum rank-one shareability 

$$
p(M_{mid}) = \frac{1 + 1}{2} = 1.
$$


In [6]:
observation_1 = rng.normal(size=(5000, 2))

# Low Observation 2
u_low = np.zeros(2)
v_low = np.zeros(2)

u_low[0] = 1.0
v_low[1] = 1.0

M_low = np.outer(u_low, v_low)
low_observation_2 = observation_1 @ M_low.T

# Mid Observation 2
u_mid = np.zeros(2)
v_mid = np.zeros(2)

u_mid[0] = 1.0
v_mid[0] = 0.5
v_mid[1] = np.sqrt(0.75)

M_mid = np.outer(u_mid, v_mid)
mid_observation_2 = observation_1 @ M_mid.T

# High Observation 2
u_high = np.zeros(2)
v_high = np.zeros(2)

u_high[0] = 1
v_high[0] = 1

M_high = np.outer(u_high, v_high)
high_observation_2 = observation_1 @ M_high.T

print("Synthetic Observations\n")
print(f"{'Data':<20}{'Shape':>12}")
print(f"{'Observation 1':<20}{str(observation_1.shape):>12}")
print(f"{'Observation 2 Low':<20}{str(low_observation_2.shape):>12}")
print(f"{'Observation 2 Mid':<20}{str(mid_observation_2.shape):>12}")
print(f"{'Observation 2 High':<20}{str(high_observation_2.shape):>12}")

Synthetic Observations

Data                       Shape
Observation 1          (5000, 2)
Observation 2 Low      (5000, 2)
Observation 2 Mid      (5000, 2)
Observation 2 High     (5000, 2)


## Establish Synthetic Data Splits

The synthetic rank-one data is divided into training, validation, and holdout test sets using a 60/20/20 split.

In [7]:
train_idx = int(observation_1.shape[0] * 0.60)
val_idx = train_idx + int(observation_1.shape[0] * 0.20)

train_observation_1 = observation_1[:train_idx]
val_observation_1 = observation_1[train_idx:val_idx]
test_observation_1 = observation_1[val_idx:]

train_low_observation_2 = low_observation_2[:train_idx]
val_low_observation_2 = low_observation_2[train_idx:val_idx]
test_low_observation_2 = low_observation_2[val_idx:]

train_mid_observation_2 = mid_observation_2[:train_idx]
val_mid_observation_2 = mid_observation_2[train_idx:val_idx]
test_mid_observation_2 = mid_observation_2[val_idx:]

train_high_observation_2 = high_observation_2[:train_idx]
val_high_observation_2 = high_observation_2[train_idx:val_idx]
test_high_observation_2 = high_observation_2[val_idx:]

print("Dataset Shapes\n")
print(f"{'Dataset':<20}{'Train':>12}{'Validation':>14}{'Test':>12}")
print(f"{'Observation 1':<20}{str(train_observation_1.shape):>12}{str(val_observation_1.shape):>14}{str(test_observation_1.shape):>12}")
print(f"{'Observation 2 Low':<20}{str(train_low_observation_2.shape):>12}{str(val_low_observation_2.shape):>14}{str(test_low_observation_2.shape):>12}")
print(f"{'Observation 2 Mid':<20}{str(train_mid_observation_2.shape):>12}{str(val_mid_observation_2.shape):>14}{str(test_mid_observation_2.shape):>12}")
print(f"{'Observation 2 High':<20}{str(train_high_observation_2.shape):>12}{str(val_high_observation_2.shape):>14}{str(test_high_observation_2.shape):>12}")

Dataset Shapes

Dataset                    Train    Validation        Test
Observation 1          (3000, 2)     (1000, 2)   (1000, 2)
Observation 2 Low      (3000, 2)     (1000, 2)   (1000, 2)
Observation 2 Mid      (3000, 2)     (1000, 2)   (1000, 2)
Observation 2 High     (3000, 2)     (1000, 2)   (1000, 2)


## Shareability Scores

Shareability scores are computed using only the training observations. The validation and holdout sets remain excluded from metric estimation so thta they can be used for model selection and held-out evaluation.

In [8]:
low_shareability = ssl_encoder_shareability(train_observation_1, train_low_observation_2)
mid_shareability = ssl_encoder_shareability(train_observation_1, train_mid_observation_2)
high_shareability = ssl_encoder_shareability(train_observation_1, train_high_observation_2)

print("Theoretical Shareability\n")
print(f"{'Case':<12}{'Shareability':>14}")
print(f"{'Low':<12}{low_shareability:>14.4f}")
print(f"{'Mid':<12}{mid_shareability:>14.4f}")
print(f"{'High':<12}{high_shareability:>14.4f}")

Theoretical Shareability

Case          Shareability
Low                 0.5144
Mid                 0.7444
High                0.9998


In [9]:
train_observation_1_tensor = torch.tensor(train_observation_1, dtype=torch.float32)
val_observation_1_tensor = torch.tensor(val_observation_1, dtype=torch.float32)
test_observation_1_tensor = torch.tensor(test_observation_1, dtype=torch.float32)

train_low_observation_2_tensor = torch.tensor(train_low_observation_2, dtype=torch.float32)
val_low_observation_2_tensor = torch.tensor(val_low_observation_2, dtype=torch.float32)
test_low_observation_2_tensor = torch.tensor(test_low_observation_2, dtype=torch.float32)

train_mid_observation_2_tensor = torch.tensor(train_mid_observation_2, dtype=torch.float32)
val_mid_observation_2_tensor = torch.tensor(val_mid_observation_2, dtype=torch.float32)
test_mid_observation_2_tensor = torch.tensor(test_mid_observation_2, dtype=torch.float32)

train_high_observation_2_tensor = torch.tensor(train_high_observation_2, dtype=torch.float32)
val_high_observation_2_tensor = torch.tensor(val_high_observation_2, dtype=torch.float32)
test_high_observation_2_tensor = torch.tensor(test_high_observation_2, dtype=torch.float32)

## Low Shareability Rank-One Case

In [10]:
low_shared_encoder = SharedEncoder(vector_size=2)
low_shared_optimizer = torch.optim.SGD(low_shared_encoder.parameters(), lr=1e-3)

epochs = 5000
best_val_loss = float("inf")
best_state = None

for i in range(epochs):
    low_shared_encoder.train()
    Z_x = low_shared_encoder(train_observation_1_tensor)
    Z_y = low_shared_encoder(train_low_observation_2_tensor)
    loss = -torch.abs(torch.mean(Z_x * Z_y))
    train_loss = loss.item()
    low_shared_optimizer.zero_grad()
    loss.backward()
    low_shared_optimizer.step()
    with torch.no_grad():
        weights = low_shared_encoder.w.weight
        weights.div_(weights.norm(p=2))
             
    low_shared_encoder.eval()
    with torch.no_grad():
        Z_x = low_shared_encoder(val_observation_1_tensor)
        Z_y = low_shared_encoder(val_low_observation_2_tensor)
        loss = -torch.abs(torch.mean(Z_x * Z_y))
        val_loss = loss.item()
            
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(low_shared_encoder.state_dict())

print(f"Low Shared Validation Loss: {best_val_loss}")
        
if best_state is not None:
    low_shared_encoder.load_state_dict(best_state)

Low Shared Validation Loss: -0.4916570782661438


In [11]:
low_seperate_encoder = SeperateEncoder(vector_size=2)
low_seperate_optimizer = torch.optim.SGD(low_seperate_encoder.parameters(), lr=1e-3)

epochs = 5000
best_val_loss = float("inf")
best_state = None

for i in range(epochs):
    low_seperate_encoder.train()
    Z_x, Z_y = low_seperate_encoder(train_observation_1_tensor, train_low_observation_2_tensor)
    loss = -torch.abs(torch.mean(Z_x * Z_y))
    train_loss = loss.item()
    low_seperate_optimizer.zero_grad()
    loss.backward()
    low_seperate_optimizer.step()
    with torch.no_grad():
        current_weights = low_seperate_encoder.a.weight
        future_weights = low_seperate_encoder.b.weight
        
        current_weights.div_(current_weights.norm(p=2)) 
        future_weights.div_(future_weights.norm(p=2))
             
    low_seperate_encoder.eval()
    with torch.no_grad():
        Z_x, Z_y = low_seperate_encoder(val_observation_1_tensor, val_low_observation_2_tensor)
        loss = -torch.abs(torch.mean(Z_x * Z_y))
        val_loss = loss.item()
            
            
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(low_seperate_encoder.state_dict())

print(f"Low Separate Validation Loss: {best_val_loss}")
        
if best_state is not None:
    low_seperate_encoder.load_state_dict(best_state)

Low Separate Validation Loss: -0.9985048770904541


In [12]:
with torch.no_grad():
    J_shared_low_train_x = low_shared_encoder(train_observation_1_tensor)
    J_shared_low_train_y = low_shared_encoder(train_low_observation_2_tensor)
    J_shared_low_train = -torch.abs(torch.mean(J_shared_low_train_x * J_shared_low_train_y))
    
    J_shared_low_val_x = low_shared_encoder(val_observation_1_tensor)
    J_shared_low_val_y = low_shared_encoder(val_low_observation_2_tensor)
    J_shared_low_val = -torch.abs(torch.mean(J_shared_low_val_x * J_shared_low_val_y))

    J_shared_low_test_x = low_shared_encoder(test_observation_1_tensor)
    J_shared_low_test_y = low_shared_encoder(test_low_observation_2_tensor)
    J_shared_low_test = -torch.abs(torch.mean(J_shared_low_test_x * J_shared_low_test_y))

with torch.no_grad():
    J_sep_low_train_x, J_sep_low_train_y = low_seperate_encoder(train_observation_1_tensor, train_low_observation_2_tensor)
    J_sep_low_train = -torch.abs(torch.mean(J_sep_low_train_x * J_sep_low_train_y))
        
    J_sep_low_val_x, J_sep_low_val_y = low_seperate_encoder(val_observation_1_tensor, val_low_observation_2_tensor)
    J_sep_low_val = -torch.abs(torch.mean(J_sep_low_val_x * J_sep_low_val_y))

    J_sep_low_test_x, J_sep_low_test_y = low_seperate_encoder(test_observation_1_tensor, test_low_observation_2_tensor)
    J_sep_low_test = -torch.abs(torch.mean(J_sep_low_test_x * J_sep_low_test_y))

print("Low Shareability Encoders\n")
print(f"{'Split':<12}{'Shared':>10}{'Separate':>10}")
print(f"{'Train':<12}{-J_shared_low_train:>10.4f}{-J_sep_low_train:>10.4f}")
print(f"{'Validation':<12}{-J_shared_low_val:>10.4f}{-J_sep_low_val:>10.4f}")
print(f"{'Test':<12}{-J_shared_low_test:>10.4f}{-J_sep_low_test:>10.4f}")

Low Shareability Encoders

Split           Shared  Separate
Train           0.5342    1.0393
Validation      0.4917    0.9985
Test            0.5656    1.1176


In [13]:
low_train = J_shared_low_train / J_sep_low_train
low_val = J_shared_low_val / J_sep_low_val
low_test = J_shared_low_test / J_sep_low_test

print("Low Shareability Ratio\n")
print(f"{'Split':<12}{'Ratio':>10}")
print(f"{'Train':<12}{low_train:>10.4f}")
print(f"{'Val':<12}{low_val:>10.4f}")
print(f"{'Test':<12}{low_test:>10.4f}")

Low Shareability Ratio

Split            Ratio
Train           0.5140
Val             0.4924
Test            0.5061


## Intermediate Shareability Rank-One Case

### Intermediate Shared Encoder Restarts

In the intermediate shared case, the objective contains two relevent stationary directions

$$
q_+ = u + v \qquad \text{and} \qquad q_- = u - v.
$$

These directions correspond to different objective magnitudes. The global maximum is associated with the direction containing the larger absolute eigenvalue, but gradient based training can converge to the lower valued stationary direction depending on initialization.

To reduce initialization dependance, the intermediate shared encoder is trained using 10 random restarts. The best validation checkpoint across restarts is retained for evaluation.

In [14]:
best_restart_val = float("inf")
best_restart_state = None
best_restart = None

for restart in range(10):
    mid_shared_encoder = SharedEncoder(vector_size=2)
    mid_shared_optimizer = torch.optim.SGD(mid_shared_encoder.parameters(), lr=1e-3)

    epochs = 5000
    best_val_loss = float("inf")
    best_state = None

    for i in range(epochs):
        train_loss = 0
        mid_shared_encoder.train()
        Z_x = mid_shared_encoder(train_observation_1_tensor)
        Z_y = mid_shared_encoder(train_mid_observation_2_tensor)
        loss = -torch.abs(torch.mean(Z_x * Z_y))
        train_loss += loss.item()
        mid_shared_optimizer.zero_grad()
        loss.backward()
        mid_shared_optimizer.step()
        with torch.no_grad():
            weights = mid_shared_encoder.w.weight
            weights.div_(weights.norm(p=2))
                
        val_loss = 0
        mid_shared_encoder.eval()
        with torch.no_grad():
            Z_x = mid_shared_encoder(val_observation_1_tensor)
            Z_y = mid_shared_encoder(val_mid_observation_2_tensor)
            loss = -torch.abs(torch.mean(Z_x * Z_y))
            val_loss += loss.item() 
                
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(mid_shared_encoder.state_dict())
        
    if best_val_loss < best_restart_val:
        best_restart_val = best_val_loss
        best_restart_state = best_state
        best_restart = restart
        
print(f"Mid Shared Restart: {best_restart}")
print(f"Mid Shared Validation Loss: {best_restart_val}")
        
if best_restart_state is not None:
        mid_shared_encoder.load_state_dict(best_restart_state)
        

        

Mid Shared Restart: 2
Mid Shared Validation Loss: -0.7366204857826233


In [15]:
mid_seperate_encoder = SeperateEncoder(vector_size=2)
mid_seperate_optimizer = torch.optim.SGD(mid_seperate_encoder.parameters(), lr=1e-3)

epochs = 5000
best_val_loss = float("inf")
best_state = None

for i in range(epochs):
    mid_seperate_encoder.train()
    Z_x, Z_y = mid_seperate_encoder(train_observation_1_tensor, train_mid_observation_2_tensor)
    loss = -torch.abs(torch.mean(Z_x * Z_y))
    train_loss = loss.item()
    mid_seperate_optimizer.zero_grad()
    loss.backward()
    mid_seperate_optimizer.step()
    with torch.no_grad():
        current_weights = mid_seperate_encoder.a.weight
        future_weights = mid_seperate_encoder.b.weight
        
        current_weights.div_(current_weights.norm(p=2)) 
        future_weights.div_(future_weights.norm(p=2))
             
    mid_seperate_encoder.eval()
    with torch.no_grad():
        Z_x, Z_y = mid_seperate_encoder(val_observation_1_tensor, val_mid_observation_2_tensor)
        loss = -torch.abs(torch.mean(Z_x * Z_y))
        val_loss = loss.item()
            
    if (val_loss) < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(mid_seperate_encoder.state_dict())
    
print(f"Mid Separate Validation Loss: {best_val_loss}")

if best_state is not None:
    mid_seperate_encoder.load_state_dict(best_state)

Mid Separate Validation Loss: -0.9860126972198486


In [16]:
with torch.no_grad():
    J_shared_mid_train_x = mid_shared_encoder(train_observation_1_tensor)
    J_shared_mid_train_y = mid_shared_encoder(train_mid_observation_2_tensor)
    J_shared_mid_train = -torch.abs(torch.mean(J_shared_mid_train_x * J_shared_mid_train_y))
    
    J_shared_mid_val_x = mid_shared_encoder(val_observation_1_tensor)
    J_shared_mid_val_y = mid_shared_encoder(val_mid_observation_2_tensor)
    J_shared_mid_val = -torch.abs(torch.mean(J_shared_mid_val_x * J_shared_mid_val_y))

    J_shared_mid_test_x = mid_shared_encoder(test_observation_1_tensor)
    J_shared_mid_test_y = mid_shared_encoder(test_mid_observation_2_tensor)
    J_shared_mid_test = -torch.abs(torch.mean(J_shared_mid_test_x * J_shared_mid_test_y))

with torch.no_grad():
    J_sep_mid_train_x, J_sep_mid_train_y = mid_seperate_encoder(train_observation_1_tensor, train_mid_observation_2_tensor)
    J_sep_mid_train = -torch.abs(torch.mean(J_sep_mid_train_x * J_sep_mid_train_y))
        
    J_sep_mid_val_x, J_sep_mid_val_y = mid_seperate_encoder(val_observation_1_tensor, val_mid_observation_2_tensor)
    J_sep_mid_val = -torch.abs(torch.mean(J_sep_mid_val_x * J_sep_mid_val_y))

    J_sep_mid_test_x, J_sep_mid_test_y = mid_seperate_encoder(test_observation_1_tensor, test_mid_observation_2_tensor)
    J_sep_mid_test = -torch.abs(torch.mean(J_sep_mid_test_x * J_sep_mid_test_y))

print("Mid Shareability Encoders\n")
print(f"{'Split':<12}{'Shared':>10}{'Separate':>10}")
print(f"{'Train':<12}{-J_shared_mid_train:>10.4f}{-J_sep_mid_train:>10.4f}")
print(f"{'Validation':<12}{-J_shared_mid_val:>10.4f}{-J_sep_mid_val:>10.4f}")
print(f"{'Test':<12}{-J_shared_mid_test:>10.4f}{-J_sep_mid_test:>10.4f}")

Mid Shareability Encoders

Split           Shared  Separate
Train           0.7806    1.0486
Validation      0.7366    0.9860
Test            0.7956    1.0938


In [17]:
mid_train = J_shared_mid_train / J_sep_mid_train
mid_val = J_shared_mid_val / J_sep_mid_val
mid_test = J_shared_mid_test / J_sep_mid_test

print("Mid Shareability Ratio\n")
print(f"{'Split':<12}{'Ratio':>10}")
print(f"{'Train':<12}{mid_train:>10.4f}")
print(f"{'Val':<12}{mid_val:>10.4f}")
print(f"{'Test':<12}{mid_test:>10.4f}")

Mid Shareability Ratio

Split            Ratio
Train           0.7444
Val             0.7471
Test            0.7274


## High Shareability Rank-One Case

In [18]:
high_shared_encoder = SharedEncoder(vector_size=2)
high_shared_optimizer = torch.optim.SGD(high_shared_encoder.parameters(), lr=1e-3)

epochs = 5000
best_val_loss = float("inf")
best_state = None

for i in range(epochs):
    high_shared_encoder.train()
    Z_x = high_shared_encoder(train_observation_1_tensor)
    Z_y = high_shared_encoder(train_high_observation_2_tensor)
    loss = -torch.abs(torch.mean(Z_x * Z_y))
    train_loss = loss.item()
    high_shared_optimizer.zero_grad()
    loss.backward()
    high_shared_optimizer.step()
    with torch.no_grad():
        weights = high_shared_encoder.w.weight
        weights.div_(weights.norm(p=2))
             
    high_shared_encoder.eval()
    with torch.no_grad():
        Z_x = high_shared_encoder(val_observation_1_tensor)
        Z_y = high_shared_encoder(val_high_observation_2_tensor)
        loss = -torch.abs(torch.mean(Z_x * Z_y))
        val_loss = loss.item()
            
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(high_shared_encoder.state_dict())

print(f"High Shared Validation Loss: {best_val_loss}")
    
if best_state is not None:
    high_shared_encoder.load_state_dict(best_state)

High Shared Validation Loss: -1.0021907091140747


In [19]:
high_seperate_encoder = SeperateEncoder(vector_size=2)
high_seperate_optimizer = torch.optim.SGD(high_seperate_encoder.parameters(), lr=1e-3)

epochs = 5000
best_val_loss = float("inf")
best_state = None

for i in range(epochs):
    high_seperate_encoder.train()
    Z_x, Z_y = high_seperate_encoder(train_observation_1_tensor, train_high_observation_2_tensor)
    loss = -torch.abs(torch.mean(Z_x * Z_y))
    train_loss = loss.item()
    high_seperate_optimizer.zero_grad()
    loss.backward()
    high_seperate_optimizer.step()
    with torch.no_grad():
        current_weights = high_seperate_encoder.a.weight
        future_weights = high_seperate_encoder.b.weight
        
        current_weights.div_(current_weights.norm(p=2)) 
        future_weights.div_(future_weights.norm(p=2))
             
    val_loss = 0
    high_seperate_encoder.eval()
    with torch.no_grad():
        Z_x, Z_y = high_seperate_encoder(val_observation_1_tensor, val_high_observation_2_tensor)
        loss = -torch.abs(torch.mean(Z_x * Z_y))
        val_loss = loss.item()
            
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(high_seperate_encoder.state_dict())

print(f"High Shared Validation Loss: {best_val_loss}")
    
if best_state is not None:
    high_seperate_encoder.load_state_dict(best_state)

High Shared Validation Loss: -1.0010217428207397


In [20]:
with torch.no_grad():
    J_shared_high_train_x = high_shared_encoder(train_observation_1_tensor)
    J_shared_high_train_y = high_shared_encoder(train_high_observation_2_tensor)
    J_shared_high_train = -torch.abs(torch.mean(J_shared_high_train_x * J_shared_high_train_y))
    
    J_shared_high_val_x = high_shared_encoder(val_observation_1_tensor)
    J_shared_high_val_y = high_shared_encoder(val_high_observation_2_tensor)
    J_shared_high_val = -torch.abs(torch.mean(J_shared_high_val_x * J_shared_high_val_y))

    J_shared_high_test_x = high_shared_encoder(test_observation_1_tensor)
    J_shared_high_test_y = high_shared_encoder(test_high_observation_2_tensor)
    J_shared_high_test = -torch.abs(torch.mean(J_shared_high_test_x * J_shared_high_test_y))

with torch.no_grad():
    J_sep_high_train_x, J_sep_high_train_y = high_seperate_encoder(train_observation_1_tensor, train_high_observation_2_tensor)
    J_sep_high_train = -torch.abs(torch.mean(J_sep_high_train_x * J_sep_high_train_y))
        
    J_sep_high_val_x, J_sep_high_val_y = high_seperate_encoder(val_observation_1_tensor, val_high_observation_2_tensor)
    J_sep_high_val = -torch.abs(torch.mean(J_sep_high_val_x * J_sep_high_val_y))

    J_sep_high_test_x, J_sep_high_test_y = high_seperate_encoder(test_observation_1_tensor, test_high_observation_2_tensor)
    J_sep_high_test = -torch.abs(torch.mean(J_sep_high_test_x * J_sep_high_test_y))

print("High Shareability Encoders\n")
print(f"{'Split':<12}{'Shared':>10}{'Separate':>10}")
print(f"{'Train':<12}{-J_shared_high_train:>10.4f}{-J_sep_high_train:>10.4f}")
print(f"{'Validation':<12}{-J_shared_high_val:>10.4f}{-J_sep_high_val:>10.4f}")
print(f"{'Test':<12}{-J_shared_high_test:>10.4f}{-J_sep_high_test:>10.4f}")



High Shareability Encoders

Split           Shared  Separate
Train           0.9729    0.9736
Validation      1.0022    1.0010
Test            0.9711    0.9711


In [21]:
high_train = J_shared_high_train / J_sep_high_train
high_val = J_shared_high_val / J_sep_high_val
high_test = J_shared_high_test / J_sep_high_test

print("High Shareability Ratio\n")
print(f"{'Split':<12}{'Ratio':>10}")
print(f"{'Train':<12}{high_train:>10.4f}")
print(f"{'Val':<12}{high_val:>10.4f}")
print(f"{'Test':<12}{high_test:>10.4f}")

High Shareability Ratio

Split            Ratio
Train           0.9993
Val             1.0012
Test            0.9999


## Rank-One Evaluation Results

Percent error between the empirical encoder ratio and theoretical shareability metric is calculated as

$$
\text{Percent Error} = \frac{|empirical - theoretical|}{theoretical} \times 100
$$

In [22]:
low_train_error = abs(low_train - low_shareability) / low_shareability * 100
mid_train_error = abs(mid_train - mid_shareability) / mid_shareability * 100
high_train_error = abs(high_train - high_shareability) / high_shareability * 100

low_val_error = abs(low_val - low_shareability) / low_shareability * 100
mid_val_error = abs(mid_val - mid_shareability) / mid_shareability * 100
high_val_error = abs(high_val - high_shareability) / high_shareability * 100

low_test_error = abs(low_test - low_shareability) / low_shareability * 100
mid_test_error = abs(mid_test - mid_shareability) / mid_shareability * 100
high_test_error = abs(high_test - high_shareability) / high_shareability * 100

print(f"Percent Error\n")
print(f"{'Split':<12}{'Low':>10}{'Mid':>10}{'High':>10}")
print(f"{'Train':<12}{low_train_error:>9.2f}%{mid_train_error:>9.2f}%{high_train_error:>9.2f}%")
print(f"{'Validation':<12}{low_val_error:>9.2f}%{mid_val_error:>9.2f}%{high_val_error:>9.2f}%")
print(f"{'Test':<12}{low_test_error:>9.2f}%{mid_test_error:>9.2f}%{high_test_error:>9.2f}%")

Percent Error

Split              Low       Mid      High
Train            0.08%     0.00%     0.05%
Validation       4.28%     0.36%     0.14%
Test             1.61%     2.28%     0.02%
